# Atelier 4.B — Self-RAG et Corrective RAG ★

**Durée : 45 min** · Projet fil rouge, incrément du Jour 4

Le RAG du Jour 2 fait confiance à sa recherche : il transmet au modèle les quatre premiers extraits, pertinents ou non. Deux patterns ajoutent un **contrôle** :

- **Corrective RAG (CRAG)** : un évaluateur juge chaque extrait retrouvé ; si aucun n'est pertinent, on **reformule** la requête et on interroge une **autre source** ; si rien ne convient, on **refuse** au lieu de générer ;
- **Self-RAG** : après la génération, on **vérifie** que la réponse est fondée sur les extraits et qu'elle répond à la question ; sinon, on régénère.

On construit ces deux boucles dans un graphe LangGraph explicite, puis on les compare au RAG du Jour 2. Les fonctions d'évaluation sont dans `agent_rag.py`.

In [ ]:
import json
from pathlib import Path
from typing import TypedDict

import pandas as pd
from langgraph.graph import END, START, StateGraph

import agent_rag as ar
import evaluation_agent as ev
import outils_llm as llm
import rag

MODELE = llm.choisir_modele()
sources = ar.construire_sources()
print("Modèle :", MODELE)

## 1. Huit questions de test

Le RAG du Jour 2 ne connaît qu'une source, la documentation. Les questions mêlent des cas qu'il traite bien, des questions de **dépannage** (dont la réponse est dans la base du support), une formulation éloignée des documents et une question **hors corpus**. Les `indices` sont les faits attendus dans la réponse.

In [ ]:
CAS = [
    {"id": "C1", "question": "Le papier recyclé PA-4R est-il moins cher si j'en commande beaucoup ?", "indices": [["5.90", "5,90"]]},
    {"id": "C2", "question": "Mon imprimante affiche le code E-21. Que dois-je faire ?", "indices": ["fusion"]},
    {"id": "C3", "question": "L'assise de notre siège ergonomique redescend toute seule. Que faire ?", "indices": ["vérin"]},
    {"id": "C4", "question": "Quand je tape un Z sur mon clavier KB-80, c'est un Y qui s'affiche.", "indices": ["Suisse"]},
    {"id": "C5", "question": "Combien coûte par an un contrat de maintenance Or pour 5 000 CHF de matériel ?", "indices": ["900"]},
    {"id": "C6", "question": "En combien de temps êtes-vous livrés à Zurich ?", "indices": ["2 jours ouvrables"]},
    {"id": "C7", "question": "Proposez-vous la location de véhicules utilitaires ?", "indices": [], "refus": True},
    {"id": "C8", "question": "Comment accéder à notre NAS quand l'adresse ns4.local ne répond plus ?", "indices": [["adresse IP", "NS Finder"]]},
]

def score(cas, texte):
    """1 si la réponse contient les faits attendus (ou refuse quand il le faut), 0 sinon."""
    if cas.get("refus"):
        return float(ev.est_refus(texte))
    return sum(ev.present(i, texte) for i in cas["indices"]) / len(cas["indices"])

In [ ]:
assistant_j2 = rag.Assistant(sources.documentation, sources.reranker, modele=MODELE)
reference_j2 = {}
for cas in CAS:
    r = assistant_j2.repondre(cas["question"])
    reference_j2[cas["id"]] = {"reponse": r.texte, "score": score(cas, r.texte), "appels": 1, "latence_s": round(r.latence_s, 1)}
    print(f"{cas['id']} score {reference_j2[cas['id']]['score']:.1f} | {r.texte[:150]}")

## 2. Évaluer les extraits : le cœur de CRAG

L'évaluateur reçoit la question et les extraits, et rend un verdict par extrait, en **un seul appel** (sortie structurée). Pour une question de dépannage, les extraits de la documentation sont jugés hors sujet ; ceux de la base du support, pertinents.

In [ ]:
question = CAS[1]["question"]
for source in ("documentation", "support"):
    extraits = sources.rechercher(source, question)
    verdicts = ar.evaluer_extraits(question, extraits, MODELE)
    print(f"== {source}")
    for e, ok in zip(extraits, verdicts):
        print(f"   {'✓' if ok else '✗'} [{ar.reference(e)}] {e.reference()[:80]}")

La part d'extraits jugés pertinents est la **context relevance** de la recherche : une mesure de sa précision, sans réponse de référence.

## 3. Le graphe CRAG + Self-RAG

```
rechercher → évaluer ─┬─ des extraits pertinents → générer → vérifier ─┬─ fondée et utile → fin
                      ├─ aucun, 1ʳᵉ tentative → reformuler → rechercher   └─ sinon, une seule fois → générer
                      └─ aucun, après reformulation → refuser
```

L'état du graphe garde la question, la requête et la source courantes, les extraits, la réponse, la vérification, le nombre d'appels au modèle et un journal des décisions.

In [ ]:
class EtatCRAG(TypedDict):
    question: str
    requete: str
    source: str
    extraits: list
    pertinents: list
    tentatives: int
    reponse: str
    verification: dict
    generations: int
    appels: int
    journal: list


def rechercher(etat):
    extraits = sources.rechercher(etat["source"], etat["requete"])
    return {"extraits": extraits, "journal": etat["journal"] + [f"recherche dans {etat['source']} : « {etat['requete']} »"]}


def evaluer(etat):
    verdicts = ar.evaluer_extraits(etat["question"], etat["extraits"], MODELE)
    pertinents = [e for e, ok in zip(etat["extraits"], verdicts) if ok]
    return {"pertinents": pertinents, "appels": etat["appels"] + 1,
            "journal": etat["journal"] + [f"évaluation : {len(pertinents)}/{len(etat['extraits'])} extraits pertinents"]}


def apres_evaluation(etat):
    if etat["pertinents"]:
        return "generer"
    return "reformuler" if etat["tentatives"] == 0 else "refuser"


def reformuler(etat):
    nouvelle = ar.reformuler(etat["question"], etat["source"], MODELE)
    return {"requete": nouvelle.requete, "source": nouvelle.source, "tentatives": etat["tentatives"] + 1,
            "appels": etat["appels"] + 1, "journal": etat["journal"] + [f"reformulation vers {nouvelle.source}"]}


def generer(etat):
    consigne = ""
    if etat["verification"]:   # deuxième génération : on signale les affirmations non fondées
        consigne = ("\nTa réponse précédente contenait des affirmations absentes des extraits : "
                    + "; ".join(etat["verification"]["affirmations_non_fondees"])
                    + ". Ne garde que ce que disent les extraits.")
    texte = ar.generer(etat["question"], etat["pertinents"], MODELE, consigne).texte
    return {"reponse": texte, "generations": etat["generations"] + 1, "appels": etat["appels"] + 1,
            "journal": etat["journal"] + [f"génération n° {etat['generations'] + 1}"]}


def verifier(etat):
    v = ar.verifier_reponse(etat["question"], etat["reponse"], etat["pertinents"], MODELE)
    return {"verification": v.model_dump(), "appels": etat["appels"] + 1,
            "journal": etat["journal"] + [f"vérification : fondée={v.fondee}, répond={v.repond}"]}


def apres_verification(etat):
    v = etat["verification"]
    if (v["fondee"] and v["repond"]) or etat["generations"] >= 2:
        return END
    return "generer"


def refuser(etat):
    return {"reponse": ar.NON_TROUVE + " Je vous invite à contacter le service client.",
            "journal": etat["journal"] + ["refus : aucune source pertinente"]}


graphe = StateGraph(EtatCRAG)
for nom, noeud in [("rechercher", rechercher), ("evaluer", evaluer), ("reformuler", reformuler),
                   ("generer", generer), ("verifier", verifier), ("refuser", refuser)]:
    graphe.add_node(nom, noeud)
graphe.add_edge(START, "rechercher")
graphe.add_edge("rechercher", "evaluer")
graphe.add_conditional_edges("evaluer", apres_evaluation, ["generer", "reformuler", "refuser"])
graphe.add_edge("reformuler", "rechercher")
graphe.add_edge("generer", "verifier")
graphe.add_conditional_edges("verifier", apres_verification, ["generer", END])
graphe.add_edge("refuser", END)
crag = graphe.compile()
print(crag.get_graph().draw_mermaid())

In [ ]:
def etat_initial(question, source="documentation"):
    return {"question": question, "requete": question, "source": source, "extraits": [], "pertinents": [],
            "tentatives": 0, "reponse": "", "verification": {}, "generations": 0, "appels": 0, "journal": []}

resultat = crag.invoke(etat_initial(CAS[1]["question"]))
print("\n".join(resultat["journal"]))
print("\n", resultat["reponse"])

Lisez le journal. La question de dépannage était posée à la documentation : si tout se passe comme prévu, l'évaluateur écarte ses extraits, la reformulation choisit la base du support et la réponse cite l'article `KB-104`. Sinon, cherchez l'étape fautive : la recherche (bons extraits absents), l'évaluateur (bons extraits écartés) ou la reformulation (mauvaise source).

## 4. Comparaison avec le RAG du Jour 2

In [ ]:
lignes = []
for cas in CAS:
    r = crag.invoke(etat_initial(cas["question"]))
    lignes.append({"id": cas["id"], "question": cas["question"][:55],
                   "score Jour 2": reference_j2[cas["id"]]["score"], "score CRAG": score(cas, r["reponse"]),
                   "appels Jour 2": 1, "appels CRAG": r["appels"],
                   "chemin": " → ".join(j.split(" :")[0] for j in r["journal"]),
                   "reponse": r["reponse"]})
comparaison = pd.DataFrame(lignes).set_index("id")
comparaison.drop(columns="reponse")

In [ ]:
bilan = {
    "score_moyen_jour2": round(comparaison["score Jour 2"].mean(), 3),
    "score_moyen_crag": round(comparaison["score CRAG"].mean(), 3),
    "appels_moyens_jour2": 1.0,
    "appels_moyens_crag": round(comparaison["appels CRAG"].mean(), 2),
}
Path("resultats").mkdir(exist_ok=True)
Path("resultats/comparaison_4B.json").write_text(json.dumps(
    {"modele": MODELE, "bilan": bilan, "details": comparaison.reset_index().to_dict(orient="records")},
    ensure_ascii=False, indent=2), encoding="utf-8")
bilan

**À observer.** Comparez question par question : CRAG doit rattraper les questions de dépannage en changeant de source, et refuser proprement hors corpus. Une régression vient souvent de l'évaluateur, qui écarte un bon extrait : relisez ses verdicts. Le prix : trois à six appels au modèle au lieu d'un. Le contrôle Self-RAG n'est utile que si l'on agit sur son verdict : régénérer, ou signaler la réponse à un humain.

## 5. Dans l'agent : la recherche corrective comme outil

L'agent de l'atelier 4.A choisit déjà sa source, mais il reçoit tous les extraits, même hors sujet. Avec `corrective=True`, ses outils de recherche passent par `ar.recherche_corrective` : les extraits non pertinents sont écartés avant d'atteindre l'agent, et une recherche infructueuse est reformulée. L'atelier 4.C mesure l'effet sur le golden set.

In [ ]:
journal = []
agent_correctif = ar.construire_agent(
    ar.creer_outils(sources, corrective=True, modele_evaluateur=MODELE, journal=journal), modele=MODELE)
r = await ar.demander(agent_correctif, "CL-10345", "L'assise d'un de nos sièges SE-700 redescend toute seule. Que pouvez-vous faire ?")
ar.afficher(r, sources)
pd.DataFrame(journal)

## À retenir

- **CRAG** contrôle la recherche : évaluer les extraits, reformuler, changer de source, refuser plutôt que générer sur du bruit.
- **Self-RAG** contrôle la génération : la réponse est-elle fondée sur les extraits, répond-elle à la question ?
- Chaque contrôle est un **appel au modèle** : on les réserve aux cas où une erreur coûte cher, et on mesure le gain sur le golden set.
- Le Self-RAG de l'article original entraîne le modèle à produire ses propres jetons de réflexion ; ici, un LLM juge joue ce rôle, sans entraînement.